In [0]:
# Load bronze tables into spark dataframes
airlines_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airlines')
airports_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airports')
flights_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.flights')
cancellation_codes_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.cancellation_codes')

In [0]:
from pyspark.sql import functions as F

**1. Standardize data types**
- convert every value into the correct data format

In [0]:
# check schema of airlines_df
airlines_df.printSchema()

# check sample if it matches
display(airlines_df.limit(5))

In [0]:
# check schema of airports_df
airports_df.printSchema()

# check sample data of airports_df
display(airports_df.limit(5))

In [0]:
# check schema of cancellation_codes_df
cancellation_codes_df.printSchema()

# check sample data of cancellation_codes_df
display(cancellation_codes_df.limit(5))

In [0]:
# check schema of flights_df
flights_df.printSchema()

# check sample data of flights_df
display(flights_df.limit(5))

In [0]:
# covert columns to their corresponding appropriate data types
flights_df = flights_df.withColumn(
    'FLIGHT_NUMBER', F.col('FLIGHT_NUMBER').cast('string')
).withColumn(
    'DIVERTED', F.col('DIVERTED').cast('boolean')
).withColumn(
    'CANCELLED', F.col('CANCELLED').cast('boolean')
)

**2. Handle missing values**
- adress null entries

In [0]:
# Show rows with null values, from bronze_ingestion
display(airports_df.filter(
    F.col('LATITUDE').isNull()
))

In [0]:
# create mapping data for missing lat and lon
# data is gathered from google maps

missing_airport_coordinates = [
    ('ECP', 30.3582, -85.7956),
    ('PBG', 44.65094, -73.46814),
    ('UST', 29.959, -81.340)
]

# turn into a dataframe
airports_ref = spark.createDataFrame(missing_airport_coordinates, schema=['IATA_CODE', 'LAT_REF', 'LON_REF'])

# join dataframes, coalesce to replace null values, drop reference columns
airports_df = airports_df.join(
                    airports_ref,
                    on='IATA_CODE',
                    how='left'
                ).withColumn(
                    'LATITUDE',
                    F.coalesce(F.col('LATITUDE'), F.col('LAT_REF'))
                ).withColumn(
                    'LONGITUDE',
                    F.coalesce(F.col('LONGITUDE'), F.col('LON_REF'))
                ).drop(
                    'LAT_REF',
                    'LON_REF'
                )

In [0]:
flights_df.printSchema()

In [0]:
# Check columns with null values on the flights dataframe
display(flights_df.select(
    [F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}') 
    for column in flights_df.columns]
))

In [0]:
# Create FLIGHT_DATE from year, month, day columns
flights_df = flights_df.withColumn(
    'FLIGHT_DATE',
    F.make_date(
        F.col('YEAR'),
        F.col('MONTH'),
        F.col('DAY')
    )
)

In [0]:
# display the 6 records with null on SCHEDULED_TIME

display(
        flights_df.filter(
        F.col('SCHEDULED_TIME').isNull()
    ).select(
        'FLIGHT_DATE',
        'ORIGIN_AIRPORT',
        'DESTINATION_AIRPORT',
        'SCHEDULED_DEPARTURE',
        'SCHEDULED_ARRIVAL',
        'SCHEDULED_TIME'
    )
)



In [0]:
# display alike records from flight_df with the same ORIGIN_AIRPORT, DESTINATION_AIRPORT, SCHEDULED_DEPARTURE, SCHEDULED_ARRIVAL
display(flights_df
 .filter(F.col('SCHEDULED_TIME').isNull())
 .alias('missing')
 .join(
     flights_df.alias('reference'),
     (
         (F.col('missing.ORIGIN_AIRPORT') == F.col('reference.ORIGIN_AIRPORT')) &
         (F.col('missing.DESTINATION_AIRPORT') == F.col('reference.DESTINATION_AIRPORT')) &
         (F.col('missing.SCHEDULED_DEPARTURE') == F.col('reference.SCHEDULED_DEPARTURE')) &
         (F.col('missing.SCHEDULED_ARRIVAL') == F.col('reference.SCHEDULED_ARRIVAL')) &
         (F.col('reference.SCHEDULED_TIME').isNotNull())
     ),
     'inner'
 )
 .select(
    F.col("missing.FLIGHT_DATE").alias("MISSING_DATE"),
    F.col("missing.ORIGIN_AIRPORT"),
    F.col("missing.DESTINATION_AIRPORT"),
    F.col("missing.SCHEDULED_DEPARTURE"),
    F.col("missing.SCHEDULED_ARRIVAL"),
    F.col("reference.FLIGHT_DATE").alias("REFERENCE_DATE"),
    F.col("reference.SCHEDULED_TIME").alias("REFERENCE_SCHEDULED_TIME")
)
 .orderBy('REFERENCE_SCHEDULED_TIME', 'MISSING_DATE')
)

In [0]:
# create mapping dataset for missing SCHEDULED_TIME values
missing_scheduled_time = [
    ('2015-02-01', 172),
    ('2015-02-10', 172),
    ('2015-04-20', 178),
    ('2015-04-26', 111),
    ('2015-05-09', 130),
    ('2015-05-10', 113)
]

# turn into dataframe
scheduled_time_reference = spark.createDataFrame(
    missing_scheduled_time, 
    ['FLIGHT_DATE', 'SCHEDULED_TIME_REF']
)

# join then coalesce to replace null values
flights_df = flights_df.join(
    scheduled_time_reference,
    on='FLIGHT_DATE',
    how='left'
).withColumn(
    'SCHEDULED_TIME',
    F.coalesce('SCHEDULED_TIME', 'SCHEDULED_TIME_REF')
).drop(
    'SCHEDULED_TIME_REF'
)

In [0]:
# Check columns with null values on the flights dataframe
display(flights_df.select(
    [F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}') 
    for column in flights_df.columns]
))

In [0]:
# check how many flights are cancelled

display(
    flights_df.groupBy(
        'CANCELLED'
    ).count()
)

In [0]:
# for cancelled flights, how many rows per column are null

display(
    flights_df.filter(
        F.col('CANCELLED') == 'true'
    ).select(
        [F.sum(F.col(c).isNull().cast('int')).alias(f'null_{c}')
        for c in flights_df.columns]
    )
)

In [0]:
# of 89,884 cancelled flights, only 86,153 does not have value on DEPARTURE_TIME and DEPARTURE_DELAY
# investigate cancelled flights that departed

display(
    flights_df.filter(
        (F.col('CANCELLED') == 'true') &
        (F.col('DEPARTURE_TIME').isNotNull() | F.col('DEPARTURE_DELAY').isNotNull())
    )
)

In [0]:
# among the departed cancelled flights, some records had a value for WHEELS_OFF and some didn't
# investigate those that ddi have WHEELS_OFF value

cancelled_after_wheels_off = flights_df.filter(
        (F.col('CANCELLED') == 'true') &
        (F.col('DEPARTURE_TIME').isNotNull()) &
        (F.col('WHEELS_OFF').isNotNull())
    )

display(cancelled_after_wheels_off)


In [0]:
# are there records where wheels_off is populated as well as other operational columns even if a flight is recorded as cancelled

display(
    cancelled_after_wheels_off.filter(
    (F.col('ELAPSED_TIME').isNotNull()) |
    (F.col('AIR_TIME').isNotNull()) |
    (F.col('WHEELS_ON').isNotNull()) |
    (F.col('TAXI_IN').isNotNull()) |
    (F.col('ARRIVAL_TIME').isNotNull()) |
    (F.col('ARRIVAL_DELAY').isNotNull())
    )
)

**Conclusions so far:**
- Cancelled flights do not necessarily mean the aircraft never left the gate.
- Some cancelled flights have `DEPARTURE_TIME`, indicating that they departed the gate before cancellation.
- Some cancelled flights progressed further and have `TAXI_OUT` and `WHEELS_OFF` values.
- `WHEELS_OFF` is the further populated operational stage observed among cancelled flights.
- Cancelled flights with `WHEELS_OFF` have null values for subsequent operational columns such as `AIR_TIME`, `WHEELS_ON`, `TAXI_IN`, `AARIVAL_TIME`, etc.

**BTS Documentation:**
According to the Bureau of Transportation Statistics, a flight may still be classified as **cancelled after wheels-off**. These cases are referred to as **fly returns**, where the aircraft takes off but subsequently returns.
- Therefore, null values in later operational columns for cancelled flights can be structurally expected and should not automatically be treated as missing-date errors.
- These structural nulls should not be imputed simply to make the columns look complete.

In [0]:
# how many cancelled flights have a departure time
cancelled_and_departed = flights_df.filter(
    (F.col('CANCELLED') == 'true') &
    (F.col('DEPARTURE_TIME').isNotNull())
).count()

# how many cancelled flights have a wheels off
cancelled_and_wheels_off = flights_df.filter(
    (F.col('CANCELLED') == 'true') &
    (F.col('WHEELS_OFF').isNotNull())
).count()

print('Cancelled and departed:', cancelled_and_departed)
print('Cancelled and wheels off:', cancelled_and_wheels_off)


In [0]:
display(
    flights_df.select(
        [F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}')
        for column in flights_df.columns]
    ))

In [0]:
# check which fields are commonly not populated in diverted flights
display(flights_df.filter(
    F.col('DIVERTED') == 'true'
))

In [0]:
display(flights_df.filter(
    (F.col('DIVERTED') == 'true') &
    (F.col('ELAPSED_TIME').isNotNull() | F.col('AIR_TIME').isNotNull() | F.col('ARRIVAL_DELAY').isNotNull())
))

# all diverted flights have no value on ELAPSED_TIME, AIR_TIME, and ARRIVAL_DELAY

In [0]:
display(flights_df.filter(
    (F.col('DIVERTED') == 'true')
    ).select(
        F.count('DIVERTED').alias('total_diverted'),
        F.sum(F.col('WHEELS_ON').isNotNull().cast('int')).alias(f'populated_WHEELS_ON'),
        F.sum(F.col('TAXI_IN').isNotNull().cast('int')).alias(f'populated_TAXI_IN'),
        F.sum(F.col('ARRIVAL_TIME').isNotNull().cast('int')).alias(f'populated_ARRIVAL_TIME'),
    )
        )

In [0]:
# since number of populated values on WHEELS_ON, TAXI_IN, and ARRIVAL_TIME is the same, investigate if they are all the same records
# total count of rows should equal to the total number of populated values

display(flights_df.filter(
    (F.col('DIVERTED') == 'true') &
    (F.col('WHEELS_ON').isNotNull() | F.col('TAXI_IN').isNotNull() | F.col('ARRIVAL_TIME').isNotNull())
).select(
    F.count('DIVERTED').alias('populated_rows')
)
        )

In [0]:
display(
    flights_df.filter(
        F.col('DIVERTED') == 'true'
    ).select(
        F.count('DIVERTED').alias('Total Diverted Flights'),
        F.sum(F.col('WHEELS_OFF').isNotNull().cast('int')).alias('Diverted Flights That Took Off')
    )
)

In [0]:
display(
    flights_df.filter(
        (F.col('DIVERTED') == 'true') & (F.col('CANCELLED') == 'true') 
    )
)

In [0]:
# why did diverted flights that definitely took off and was not cancelled have no WHEELS_ON, TAXI_IN, or ARRIVAL_TIME
display(flights_df.limit(1))

**2. Handle missing values**

In [0]:
# create mapping data for missing lat and lon
# data is gathered from google maps

missing_airport_coordinates = [
    ('ECP', 30.3582, -85.7956),
    ('PBG', 44.65094, -73.46814),
    ('UST', 29.959, 81.340)
]

In [0]:
# create a dataframe based on the missing_airport_coordinates list
airport_reference_df = spark.createDataFrame(
    missing_airport_coordinates, 
    ['IATA_CODE', 'LATITUDE_REF', 'LONGITUDE_REF']
)

In [0]:
# join two dataframes based on their iata_code
airports_df = airports_df.join(
    airport_reference_df,
    on='IATA_CODE',
    how='left'
)

In [0]:
# coalesce values
airports_df = airports_df.withColumn(
        'LATITUDE',
        F.coalesce(
            F.col('LATITUDE'),
            F.col('LATITUDE_REF')
        )
    ).withColumn(
        'LONGITUDE',
        F.coalesce(
            F.col('LONGITUDE'),
            F.col('LONGITUDE_REF')
    )
    )

In [0]:
# drop reference columns
airports_df = airports_df.drop(
    'LATITUDE_REF',
    'LONGITUDE_REF'
)

In [0]:
# Check columns with null values on the flights dataframe
display(flights_df.select(
    [F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}') 
    for column in flights_df.columns]
))

In [0]:
display(
    flights_df.withColumn(
        "SCHEDULED_TIME",
        F.when(
            F.col("SCHEDULED_TIME").isNull(),
            (
                (
                    F.unix_timestamp("SCHEDULED_ARRIVAL_DATETIME")
                    - F.unix_timestamp("SCHEDULED_DEPARTURE_DATETIME")
                ) / 60
            )
        ).otherwise(F.col("SCHEDULED_TIME"))
    )
)

In [0]:
# Check duplicates
dfs = {
    'airports_df': airports_df,
    'airlines_df': airlines_df,
    'flights_df': flights_df,
    'cancellation_codes_df': cancellation_codes_df
}

for name, df in dfs.items():
    print(f'Name: {name} | Duplicates: {df.count() - df.dropDuplicates().count()}')

In [0]:
display(flights_df.limit(10))

In [0]:
# Fill missing values pt 2
display(flights_df.fillna(
    0,
    subset = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
))